In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, auc
import os
import pickle

import scanpy as sc 
import json
import re
import pyranges as pr
from cellgrn.utils import enhancer_eval, eval_gene_peak, eval_tf_recovery, eval_tf_recovery_ctx, eval_tf_gene, load_scenic2, load_linger_ctx,load_linger_all,load_thres_grn,eval_gene_peak2,bench_sumamry

In [ ]:
with open("~/multireg/cellGRN/eval/backbone_res.pkl", "rb") as f:
    soft_res = pickle.load(f)

In [3]:
soft_res.keys()

dict_keys(['FigR', 'Pando', 'celloracle', 'GLUE', 'SCENIC+', 'LINGER'])

In [ ]:
print("hello")

In [52]:
# 1. enhancer eval
cd4_gold = pd.read_csv("/home/shaliu_fu/multireg/benchmark/datasets/pbmc/10X_PBMC_CD4_STARR.bed",sep='\t',header=None)
cd4_gold['Peak'] = cd4_gold.apply(lambda row:f"{row[0]}:{row[1]}-{row[2]}", axis=1)

In [53]:
pr_curve = pd.DataFrame()
res_summary = []
# bg_summary = []


for soft in soft_res.keys():
    gene_peak_res = soft_res[soft]['gene_peak_res']

    if gene_peak_res is not None:
        pr_table,pr_auc,epr,f1  = enhancer_eval(cd4_gold, gene_peak_res,soft)
        pr_curve = pd.concat([pr_curve,pr_table],axis=0)
        res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])
    else:
        res_summary.append([soft,'NA','NA','NA'])
res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label.loc[label > 1] = 1
/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:230: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a

In [54]:
res_summary

,method,PRAUC,EPR,F1
0,FigR,0.01397,0.01132,0.02766
1,Pando,0.03097,0.05393,0.08105
2,celloracle,NA,NA,NA
3,GLUE,0.00952,0.00799,0.02557
4,SCENIC+,0.01998,0.02796,0.04336
5,LINGER,0.07856,0.1245,0.15703


In [67]:
def bench_sumamry(df: pd.DataFrame, label_cols=None) -> pd.DataFrame:

    df = df.copy()

    if label_cols is None:
        label_cols = [df.columns[0]]

    value_cols = [c for c in df.columns if c not in label_cols]

    # ── Step 1: 按列 0-1 归一化 ──────────────────────────────────────
    df_norm = pd.DataFrame(index=df.index)
    for col in value_cols:
        series = pd.to_numeric(df[col], errors='coerce')   # 强制转数值，非数值→NaN
        col_min = series.min(skipna=True)
        col_max = series.max(skipna=True)

        if pd.isna(col_min) or pd.isna(col_max) or col_max == col_min:
            df_norm[col] = np.nan
        else:
            df_norm[col] = (series - col_min) / (col_max - col_min)

    # ── Step 2: 按行取均值 → Score 列 ────────────────────────────────
    df_out = df[label_cols].copy()
    df_out['Score'] = df_norm.mean(axis=1, skipna=True)

    return df_out

In [68]:
backbone_enhancer = bench_sumamry(res_summary,['method'])

In [80]:
eval_summary = backbone_enhancer
eval_summary.rename(columns={"Score": "enhancer_pred"},inplace=True)

In [81]:
from scipy.sparse import load_npz

gene_peak_link = load_npz('/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/gene_peak_dist_all.npz')
input_gene = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_gene.txt")]
input_peak = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/input_data/all_gene/10X_PBMC/input_peak.txt")]
gene_peak_coo = gene_peak_link.tocoo()
peak_idx = gene_peak_coo.col   # [n_nonzero]
gene_idx = gene_peak_coo.row   # [n_nonzero]

gene_peak_dist = pd.DataFrame({"Gene" : pd.Series(input_gene).values[gene_idx],
                              "Peak" : pd.Series(input_peak).values[peak_idx],
                              "Dist": gene_peak_coo.data
})
gene_peak_dist['Peak'] = gene_peak_dist['Peak'].str.replace(r'^([^-\s]+)-', r'\1:', regex=True)

In [82]:
ctx_dict = {"CD4 T":"cd4","B":"b","CD8 T":"cd8","NK":"nk"}

res_summary = []
# bg_summary = []

# range_summary = pd.DataFrame()
# pr_curve = pd.DataFrame()

for ctx in ctx_dict.keys():
    hic_gold = f"/home/shaliu_fu/multireg/benchmark/datasets/pbmc/encode_hic/{ctx_dict[ctx]}_gene_peak_gold.bed"
    gold_pr_region = pr.read_bed(hic_gold)  
    for soft in soft_res.keys():

        gene_peak_res = soft_res[soft]['gene_peak_res']
        if gene_peak_res is not None:
            gene_peak_res_ = pd.merge(gene_peak_res,gene_peak_dist,on=["Gene","Peak"],how="left")
            # soft_ = f"{soft}_{ctx}"
            print(ctx)
            print(soft)
            

            _,pr_auc,epr,f1 = eval_gene_peak2(gold_pr_region, gene_peak_res_, soft)
            # pr_table['celltype']=ctx
            # pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5),ctx])

            # range_res['celltype']=ctx
            # range_summary = pd.concat([range_summary,range_res], axis=0)
        else:
            res_summary.append([soft,'NA','NA','NA',ctx])
res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1","celltype"]

CD4 T
FigR


/tmp/ipykernel_328630/2253425135.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positi

424
0
-0.9930497419110023
CD4 T
Pando


/tmp/ipykernel_328630/2253425135.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positi

477
0
2.0303964704329487
CD4 T
GLUE


...

150
0
-0.33692437410354614
NK
SCENIC+


/tmp/ipykernel_328630/2253425135.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positi

2881
0
-1.0
NK
LINGER


/tmp/ipykernel_328630/2253425135.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['peak_gene'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}_{row[3]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:27: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  peak_gene_sel['Peak'] = peak_gene_sel.apply(lambda row: f"{row[0]}:{row[1]}-{row[2]}", axis=1)
/tmp/ipykernel_328630/2253425135.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by positi

3212
0
-0.9216580808216327


In [ ]:
backbone_gene_peak = bench_sumamry(res_summary,['method','celltype'])
backbone_gene_peak = backbone_gene_peak.groupby("method").mean("Score")


In [99]:
eval_summary =  pd.merge(eval_summary,backbone_gene_peak, on='method',how="left")
eval_summary.rename(columns={"Score": "gene_peak_pred"},inplace=True)

In [100]:
eval_summary

,method,enhancer_pred,gene_peak_pred
0,FigR,0.036312,0.081070
1,Pando,0.375673,0.068166
2,celloracle,NaN,NaN
3,GLUE,0.000000,0.032711
4,SCENIC+,0.152745,0.353914
5,LINGER,1.000000,0.807583


In [101]:
with open("/home/shaliu_fu/multireg/multigrn/input_data/gold_dataset/pbmc_gold.pkl", "rb") as f:
    gold_data = pickle.load(f)

# 评估tf recovery, PBMC总体
tf_gold = {
    "B": [
        "PAX5",    # B细胞谱系决定核心因子
        "EBF1",    # 早期B细胞发育关键因子
        "POU2F2",  # (Oct-2) 调节免疫球蛋白基因表达
        "BCL6",    # 生发中心B细胞标志
        "IRF4"     # 浆细胞分化关键因子
    ],
    "CD4 T": [
        "TCF7",    # (TCF-1) 幼稚/记忆状态维持
        "LEF1",    # 幼稚T细胞标志
        "TBX21",   # (T-bet) Th1亚群标志
        "GATA3",   # Th2亚群标志
        "FOXP3",   # Treg(调节性T细胞)核心标志
        "RORC"     # (RORγt) Th17亚群标志
    ],
    "CD8 T": [
        "RUNX3",   # CD8+ 谱系决定因子
        "EOMES",   # 效应与记忆功能调控
        "TBX21",   # (T-bet) 细胞毒性功能调控
        "PRDM1"    # (Blimp-1) 终末分化效应细胞标志
    ],
    "DC": [
        "TCF4",    # (E2-2) pDC(浆细胞样DC)特异性标志
        "IRF8",    # cDC1 和 pDC 发育关键因子
        "BATF3",   # cDC1 亚型特异性因子
        "IRF4",    # cDC2 亚型相关因子
        "ZEB2"     # 调控DC发育与分化
    ],
    "Mono": [
        "SPI1",    # (PU.1) 髓系发育主控因子
        "CEBPB",   # (C/EBPβ) 非经典单核细胞(CD16+)核心因子
        "MAFB",    # 单核/巨噬细胞分化标志
        "KLF4",    # 经典单核细胞(CD14+)相关因子
        "IRF8"     # 影响单核细胞向DC或巨噬细胞的分化
    ],
    "NK": [
        "EOMES",   # NK细胞发育与成熟关键因子
        "TBX21",   # (T-bet) 调控NK细胞毒性分子表达
        "ID2",     # 抑制T/B谱系，促进NK发育
        "IKZF3"    # (Aiolos) 调节NK细胞功能
    ]
}
tf_knock_gold = gold_data['tf_gene_gold']
tf_baseline = gold_data['tf_gene_baseline'].copy()

tf_baseline = tf_baseline[tf_baseline['TF']!=tf_baseline['Gene']]
tf_baseline = tf_baseline.nlargest(10000,"Score")

In [103]:

rec_summary = pd.DataFrame()


tf_rec_ctx, tf_rec_base = eval_tf_recovery(grn_res=tf_baseline,gold_data=tf_gold,label="Pearson",log=False) # ctx dataframe
# rec_summary.append(["Pearson",round(tf_rec_base.values[0][0],5),round(tf_rec_base.values[0][1],5)])
rec_summary = pd.concat([rec_summary,tf_rec_base],axis=0)



for soft in soft_res.keys():
# for soft in ['FigR']:

# for soft in spa_res.keys():
#     gene_peak_res = spa_res[soft]
    tf_gene_res = soft_res[soft]['grn_res']
    tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
    tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
    # tf_gene_res.drop_duplicates(inplace=True) # 合并各个ctx结果
    if tf_gene_res is not None:
        
        tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
        tf_rec_ctx, tf_rec_res = eval_tf_recovery(grn_res=tf_gene_res2,gold_data=tf_gold,label=soft,log=False) # ctx dataframe
        # rec_summary.append([soft,round(tf_rec_res.values[0][0],5),round(tf_rec_res.values[0][1],5)])
        rec_summary = pd.concat([rec_summary,tf_rec_res],axis=0)


In [121]:
tmp = rec_summary.iloc[1:,:3]

In [122]:
tmp['method'] = tmp.index.values

In [ ]:
backbone_tf = pd.DataFrame(bench_sumamry(tmp,['method']))

In [134]:
backbone_tf.reset_index(drop=True,inplace=True)

In [135]:
eval_summary =  pd.merge(eval_summary,backbone_tf, on='method',how="left")
eval_summary.rename(columns={"Score": "tf_pred"},inplace=True)

In [136]:
eval_summary

,method,enhancer_pred,gene_peak_pred,tf_pred
0,FigR,0.036312,0.081070,0.908880
1,Pando,0.375673,0.068166,0.093248
2,celloracle,NaN,NaN,0.875706
3,GLUE,0.000000,0.032711,0.255861
4,SCENIC+,0.152745,0.353914,0.645312
5,LINGER,1.000000,0.807583,0.413907


In [137]:
tmp = tf_knock_gold['PBMC_TF_knock']
sel_tf = set([i.split("_")[0] for i in tmp])
sel_tf

tf_knock_gold2 = {}
for tfs in sel_tf:
    tf_knock_gold2[tfs] = []

for i in tf_knock_gold['PBMC_TF_knock']:
    tf = i.split("_")[0]
    gene = i.split("_")[1]
    tf_knock_gold2[tf].append(gene)

In [138]:
# tmp = tf_knock_gold['PBMC_TF_knock']
# sel_tf = set([i.split("_")[0] for i in tmp])
tf_baseline = gold_data['tf_gene_baseline'].copy()

pr_curve = pd.DataFrame()
res_summary = []
# 评估tf-gene

for t_gold in tf_knock_gold.keys():
    tf_knock = tf_knock_gold[t_gold]
    
    tf_baseline2 = tf_baseline[tf_baseline["TF"].isin(sel_tf)]
    
    tf_baseline2 = tf_baseline2[tf_baseline2['TF']!=tf_baseline2['Gene']]
    
    

    pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_baseline2.nlargest(10000,"Score"),gold_data=tf_knock,
                    all_comb=tf_baseline2.shape[0],label="Pearson" ) # 
    # pr_table['celltype'] = ctx
    pr_curve = pd.concat([pr_curve,pr_table],axis=0)
    res_summary.append(['Pearson',round(pr_auc,5),round(epr,5),round(f1,5)])
    # pr_summary.append(["Pearson",t_gold, round(pr_auc,5)])
    # epr_summary.append(["Pearson",t_gold, round(epr,5)])
    for soft in soft_res.keys():
    # for soft in ['FigR']:

    # for soft in spa_res.keys():
    #     gene_peak_res = spa_res[soft]
        tf_gene_res = soft_res[soft]['grn_res']
        if tf_gene_res is not None:
            tf_gene_res = tf_gene_res[["TF","Gene","Score"]]
            tf_gene_res = tf_gene_res.groupby(['TF', 'Gene'])['Score'].max().reset_index() #只保留最高的。
            
            # tf_gene_res2 = tf_gene_res.nlargest(10000,"Score")
            tf_gene_res2 = tf_gene_res[tf_gene_res['TF'].isin(sel_tf)]
            tf_gene_res2 =  tf_gene_res2.nlargest(10000,"Score")
            print(f"{t_gold}: {soft}- candidates : {tf_gene_res2.shape[0]}")
            if tf_gene_res2.shape[0] > 1:
                pr_auc,epr,f1, pr_table = eval_tf_gene(grn_res=tf_gene_res2,gold_data=tf_knock,
                            all_comb=tf_baseline2.shape[0],label=soft ) # 
                # pr_table['celltype'] = ctx
            else:
                pr_auc = 0
                epr = 0
                f1 = 0
                pr_table = None
            pr_curve = pd.concat([pr_curve,pr_table],axis=0)
            res_summary.append([soft,round(pr_auc,5),round(epr,5),round(f1,5)])

res_summary = pd.DataFrame(res_summary)
res_summary.columns = ["method","PRAUC","EPR","F1"]

/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)


PBMC_TF_knock: FigR- candidates : 34
PBMC_TF_knock: Pando- candidates : 588
PBMC_TF_knock: celloracle- candidates : 218
PBMC_TF_knock: GLUE- candidates : 0


/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)
/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, intege

PBMC_TF_knock: SCENIC+- candidates : 425
PBMC_TF_knock: LINGER- candidates : 1786


/home/shaliu_fu/miniconda3/envs/cellgrn/lib/python3.12/site-packages/cellgrn/utils.py:536: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  soft_pred['pair'] = soft_pred.apply(lambda row: f"{row[0]}_{row[1]}", axis=1)


In [139]:
res_summary

,method,PRAUC,EPR,F1
0,Pearson,0.03870,0.04459,0.06652
1,FigR,0.06134,0.00041,0.06652
2,Pando,0.05127,0.00542,0.06652
3,celloracle,0.04243,0.00149,0.06652
4,GLUE,0.00000,0.00000,0.00000
5,SCENIC+,0.05267,0.00407,0.06652
6,LINGER,0.05567,0.01843,0.06652


In [140]:
backbone_tf_gene = pd.DataFrame(bench_sumamry(res_summary,['method']))

In [141]:
backbone_tf_gene

,method,Score
0,Pearson,0.876970
1,FigR,0.669732
2,Pando,0.652462
3,celloracle,0.575045
4,GLUE,0.000000
5,SCENIC+,0.649978
6,LINGER,0.773629


In [142]:
eval_summary =  pd.merge(eval_summary,backbone_tf_gene.iloc[1:,:], on='method',how="left")
eval_summary.rename(columns={"Score": "tf_gene_pred"},inplace=True)

In [144]:
eval_summary.to_csv("~/multireg/cellGRN/eval/results/backbone_res.csv",header=True,index=None)